In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/Hard Hat Workers"
TRAIN_IMG = f"{BASE}/train"
TRAIN_ANN = f"{BASE}/train/annotations.json"
TEST_IMG  = f"{BASE}/test"
TEST_ANN  = f"{BASE}/test/annotations.json"
MODEL_SAVE = "fasterrcnn_hardhat_best.pth"

In [ ]:
import os, json
import torch
import torchvision
from torchvision import transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image

TARGET_CLASSES = ['head', 'helmet']
NUM_CLASSES = len(TARGET_CLASSES) + 1

class FilteredCOCODataset(Dataset):
    def __init__(self, ann_file, img_dir, target_classes, transforms=None):
        self.img_dir = img_dir
        self.transforms = transforms
        with open(ann_file) as f:
            data = json.load(f)

        self.images = {img['id']: img for img in data['images']}
        self.categories = {cat['id']: cat['name'] for cat in data['categories']}

        self.target_cat_ids = {
            cat_id for cat_id, name in self.categories.items()
            if name in target_classes
        }
        name_to_new = {name: i+1 for i, name in enumerate(target_classes)}
        self.cat_id_mapping = {
            cat_id: name_to_new[self.categories[cat_id]]
            for cat_id in self.target_cat_ids
        }


        self.img_to_anns = {}
        for ann in data['annotations']:
            if ann['category_id'] in self.target_cat_ids:
                img_id = ann['image_id']
                self.img_to_anns.setdefault(img_id, []).append(ann)


        self.valid_ids = [i for i in self.images if i in self.img_to_anns]

    def __len__(self):
        return len(self.valid_ids)

    def __getitem__(self, idx):
        img_id = self.valid_ids[idx]
        info = self.images[img_id]
        img_path = os.path.join(self.img_dir, info['file_name'])
        image = Image.open(img_path).convert("RGB")

        anns = self.img_to_anns[img_id]
        boxes, labels = [], []
        for a in anns:
            x, y, w, h = a['bbox']
            boxes.append([x, y, x+w, y+h])
            labels.append(self.cat_id_mapping[a['category_id']])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([img_id]),
            "area": (boxes[:, 3]-boxes[:, 1]) * (boxes[:, 2]-boxes[:, 0]),
            "iscrowd": torch.zeros(len(boxes), dtype=torch.int64)
        }

        if self.transforms:
            image = self.transforms(image)
        return image, target

def get_transform(train=True):
    tr = [T.ToTensor()]
    if train:
        tr.append(T.RandomHorizontalFlip(0.5))
    return T.Compose(tr)

In [ ]:
full_ds = FilteredCOCODataset(TRAIN_ANN, TRAIN_IMG, TARGET_CLASSES,
                              transforms=get_transform(True))

train_size = int(0.8 * len(full_ds))
val_size = len(full_ds) - train_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size],
                                generator=torch.Generator().manual_seed(42))

def collate_fn(batch):
    return tuple(zip(*batch))

BATCH = 4
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=2, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                          num_workers=2, collate_fn=collate_fn)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

In [ ]:
from tqdm import tqdm
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = fasterrcnn_resnet50_fpn(pretrained=True)
in_feat = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, NUM_CLASSES)
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

def train_epoch(model, loader, epoch):
    model.train()
    total = 0
    loop = tqdm(loader, desc=f"Epoch {epoch} Train")
    for imgs, targets in loop:
        imgs = [i.to(device) for i in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(imgs, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
        loop.set_postfix(loss=loss.item())
    return total / len(loader)

@torch.no_grad()
def val_epoch(model, loader):
    model.train()
    total = 0
    loop = tqdm(loader, desc="Val")
    for imgs, targets in loop:
        imgs = [i.to(device) for i in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(imgs, targets)
        loss = sum(loss_dict.values())
        total += loss.item()
        loop.set_postfix(loss=loss.item())
    return total / len(loader)

best = float('inf')
for epoch in range(1, 50):
    t0 = time.time()
    train_loss = train_epoch(model, train_loader, epoch)
    val_loss = val_epoch(model, val_loader)
    t1 = time.time()
    print(f"Epoch {epoch}  Train Loss: {train_loss:.4f}  Val Loss: {val_loss:.4f}  Time: {t1-t0:.1f}s")
    if val_loss < best:
        best = val_loss
        torch.save(model.state_dict(), MODEL_SAVE)